# Polymarket sports: calibration

This notebook uses the locally saved joined analysis dataset. It compares predicted probabilities with the fraction of events that occurred, using **10 percentage point bins**. Each market is counted at most once per bin, but may appear in several bins. Predictions use the 23 hourly intervals before the final hour; the last hour before close is excluded. Sport labels were inferred from event information.

The dataset is local and is not included in Git. Run this notebook from the repository root.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("Data/all_closed_sports/analysis_cache/course_hourly_90e594f4a8a64fae.pkl")
if not DATA.exists():
    raise FileNotFoundError(f"Analysis dataset not found: {DATA.resolve()}")
df = pd.read_pickle(DATA)
print(f"{len(df):,} market-bin observations; {df.market_id.nunique():,} markets")
df.head()

## Calibration across all sports

For each bin, the horizontal coordinate is the average market probability and the vertical coordinate is the fraction of outcomes that occurred. The dashed diagonal represents perfect calibration. The orange line is a weighted linear regression through the bin averages, weighted by the number of observations in each bin.

Error bars are 95% Wilson intervals for each observed fraction. They treat observations as independent; related markets on the same event may make uncertainty larger. The regression is descriptive and uses the same data as the points.

In [ ]:
data = df.dropna(subset=["market_id", "probability_bin", "probability", "actual"]).copy()
data = data[data["probability"].between(0, 1) & data["actual"].isin([0, 1])]
if data.duplicated(["market_id", "probability_bin"]).any():
    raise ValueError("Each market should appear at most once per probability bin.")

bins = (data.groupby("probability_bin", observed=True)
        .agg(mean_probability=("probability", "mean"),
             observed_rate=("actual", "mean"), n=("actual", "size"))
        .reset_index().sort_values("mean_probability"))

# Wilson interval for a binomial proportion.
z = 1.959963984540054
p, count = bins.observed_rate, bins.n
denominator = 1 + z**2 / count
centre = (p + z**2 / (2 * count)) / denominator
half_width = z * np.sqrt(p * (1 - p) / count + z**2 / (4 * count**2)) / denominator
bins["ci_low"] = (centre - half_width).clip(0, 1)
bins["ci_high"] = (centre + half_width).clip(0, 1)
# Clip tiny floating-point errors at zero for matplotlib.
yerr = np.maximum(0, np.array([p - bins.ci_low, bins.ci_high - p]))

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0, 1], [0, 1], '--', color='0.45', label='Perfect calibration')
if len(bins) >= 2:
    slope, intercept = np.polyfit(bins.mean_probability, bins.observed_rate, 1, w=np.sqrt(bins.n))
    grid = np.linspace(0, 1, 200)
    ax.plot(grid, intercept + slope * grid, color='#c06b32', label=f'Weighted regression (slope = {slope:.2f})')
ax.errorbar(bins.mean_probability, bins.observed_rate, yerr=yerr, fmt='o',
            color='#3569a8', markersize=5, capsize=4, label='10% bin mean and Wilson 95% CI')
for row in bins.itertuples():
    ax.annotate(f'n={row.n:,}', (row.mean_probability, row.observed_rate),
                xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Predicted probability',
       ylabel='Observed event frequency',
       title=f'Sports market calibration ({data.market_id.nunique():,} markets)')
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
plt.show()
bins